In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'Getting started.pdf', 'filename-1 (1).pdf', 'Copy of filename-1.pdf', 'ya bois project.docx', 'super special awesome.docx', 'MO CHEANTAR.docx', 'MO CHEANTAR - Google Docs.pdf', 'English CBA 2019.docx', 'irish', 'romeo the madlad.docx', 'letter for irish.docx', 'MO CHEANTAR.gdoc', 'Listen.docx', 'cba business.pptx', 'CBA 2 business.docx', 'cba 2 english.docx', 'science cba 2.docx', 'dear benvolio.docx', 'daily routine orders 11.docx', 'business.docx', 'Untitled document (17).gdoc', 'Untitled document (16).gdoc', 'MRB Darren o Fríl.gdoc', 'Untitled presentation (3).gslides', 'Science fair.gslides', 'Untitled presentation (2).gslides', 'Untitled presentation (1).gslides', 'michael-reeves-offlinetv.jpg', 'VID_20200226_202150.mp4', 'Untitled presentation.gslides', 'tardis pencil pot.SLDPRT', 'filename-1.pdf', 'Leonardo da Vinci - You tube revision.docx', 'Untitled document (15).gdoc', 'Untitled document (14).gdoc', 'Untitled document (13).gdoc', 'Untitled document (12)

In [ ]:
import os
from PIL import Image
import torchvision.transforms as T

input_root = "/content/drive/MyDrive/Cattely-Cattle-Face-Images-Dataset-main"


output_root = "/content/drive/MyDrive/Cattely-Augmented"

os.makedirs(output_root, exist_ok=True)

rotate = T.RandomRotation(8)
color = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15)
blur = T.GaussianBlur(kernel_size=3)

for cow in os.listdir(input_root):

    cow_input_path = os.path.join(input_root, cow)

    if not os.path.isdir(cow_input_path):
        continue

    cow_output_path = os.path.join(output_root, cow)
    os.makedirs(cow_output_path, exist_ok=True)

    for img_name in os.listdir(cow_input_path):

        if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img_path = os.path.join(cow_input_path, img_name)
        img = Image.open(img_path).convert("RGB")

        base_name = os.path.splitext(img_name)[0]

        #saving original
        img.save(os.path.join(cow_output_path, f"{base_name}_orig.jpg"))

        #rotated
        rotated_img = rotate(img)
        rotated_img.save(os.path.join(cow_output_path, f"{base_name}_rot.jpg"))

        #colour jitter
        color_img = color(img)
        color_img.save(os.path.join(cow_output_path, f"{base_name}_color.jpg"))

        #blur
        blur_img = blur(img)
        blur_img.save(os.path.join(cow_output_path, f"{base_name}_blur.jpg"))

print("Augmentation complete")



Augmentation complete


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

data_path = "/content/drive/MyDrive/Cattely-Cattle-Face-Images-Dataset-main"

dataset = CattelyDataset(data_path)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

num_classes = len(dataset.class_to_idx)

print("Number of classes:", num_classes)
print("Number of images:", len(dataset))

model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}, Accuracy: {accuracy:.2f}%")

Using device: cuda
Number of classes: 47
Number of images: 1286
Epoch 1, Loss: 0.8615, Accuracy: 83.83%
Epoch 2, Loss: 0.0421, Accuracy: 99.69%
Epoch 3, Loss: 0.0144, Accuracy: 99.92%
Epoch 4, Loss: 0.0078, Accuracy: 100.00%
Epoch 5, Loss: 0.0056, Accuracy: 100.00%
